In [1]:
import sys
import os

# Add the parent folder to sys.path so Python can find 'scripts'
sys.path.append(os.path.abspath(".."))

# Now you can import
from scripts.data_cleaning import clean_data

In [2]:
import pandas as pd

# Go up one level then into data folder
df = pd.read_csv("../data/train.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [12]:
import pandas as pd
from scripts.data_cleaning import clean_data

# Load dataset
df = pd.read_csv("../data/train.csv")

# Clean dataset using reusable function
df = clean_data(df)

# Check cleaned dataset
df.head()
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     891 non-null    str    
 12  Age_missing  891 non-null    int64  
 13  Deck         891 non-null    str    
dtypes: float64(2), int64(6), str(6)
memory usage: 97.6 KB


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
Age_missing      0
Deck             0
dtype: int64

In [4]:
import pandas as pd
from scripts.data_cleaning import clean_data
from scripts.feature_engineering import engineer_features

# Load dataset
df = pd.read_csv("../data/train.csv")

# Clean data
df = clean_data(df)

# Engineer features
df = engineer_features(df)

# Check new features
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   PassengerId    891 non-null    int64  
 1   Survived       891 non-null    int64  
 2   Pclass         891 non-null    int64  
 3   Name           891 non-null    str    
 4   Sex            891 non-null    str    
 5   Age            891 non-null    float64
 6   SibSp          891 non-null    int64  
 7   Parch          891 non-null    int64  
 8   Ticket         891 non-null    str    
 9   Fare           891 non-null    float64
 10  Cabin          204 non-null    str    
 11  Embarked       891 non-null    str    
 12  Age_missing    891 non-null    int64  
 13  Deck           891 non-null    str    
 14  FamilySize     891 non-null    int64  
 15  IsAlone        891 non-null    int64  
 16  Title          891 non-null    str    
 17  AgeGroup       891 non-null    str    
 18  FarePerPerson  891 no

In [5]:
df.to_csv("../data/train_features.csv", index=False)

In [15]:
# Drop columns that are not numeric
non_numeric_cols = df.select_dtypes(include=['object']).columns
df.drop(columns=non_numeric_cols, inplace=True)

In [7]:
from scripts.feature_selection import feature_importance

importance = feature_importance(df, target='Survived')

# Display top 10 features
print(importance.head(10))

PassengerId      0.255897
Age              0.203484
Fare             0.185096
FarePerPerson    0.176130
Pclass           0.061277
FamilySize       0.041446
SibSp            0.024779
Parch            0.020299
Age_missing      0.019305
IsAlone          0.012287
dtype: float64


In [16]:
# Keep only numeric columns for Random Forest
df_numeric = df.select_dtypes(include=['int64','float64'])
df_numeric['Survived'] = df['Survived']  # Add target column back

In [9]:
from scripts.feature_selection import feature_importance

importance = feature_importance(df_numeric, target='Survived')

# Display top 10 features
print(importance.head(10))

PassengerId      0.255897
Age              0.203484
Fare             0.185096
FarePerPerson    0.176130
Pclass           0.061277
FamilySize       0.041446
SibSp            0.024779
Parch            0.020299
Age_missing      0.019305
IsAlone          0.012287
dtype: float64


In [10]:
# Keep features with importance > 0.01
selected_features = importance[importance > 0.01].index.tolist()

df_selected = df_numeric[selected_features + ['Survived']]  # Include target
df_selected.head()

,PassengerId,Age,Fare,FarePerPerson,Pclass,FamilySize,SibSp,Parch,Age_missing,IsAlone,Survived
0,1,22.0,7.2500,3.62500,3,2,1,0,0,0,0
1,2,38.0,71.2833,35.64165,1,2,1,0,0,0,1
2,3,26.0,7.9250,7.92500,3,1,0,0,0,1,1
3,4,35.0,53.1000,26.55000,1,2,1,0,0,0,1
4,5,35.0,8.0500,8.05000,3,1,0,0,0,1,0


In [11]:
df_selected.to_csv("../data/train_selected.csv", index=False)